# Camera Array Processing Examples

Reproducible examples for the camera-array processing tasks used in the accompanying IEEE Transactions on Computational Imaging work. The notebook is organized so each numbered task can be run independently after **Setup** and **Shared utilities**.

## Repository layout

```text
.
├── Camera_Array_Processing_Examples.ipynb
├── requirements.txt
└── data/
    ├── paired_frames/
    ├── polarization/
    ├── sweeps/arducam/
    ├── sweeps/vimba/
    ├── synchronized/color/
    └── synchronized/mono/
```

Start Jupyter from the repository root, install the dependencies with `pip install -r requirements.txt`, and run the cells for the desired task. GPU acceleration is used automatically when CUDA is available.


## Setup


In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pywt
import torch
import torch.nn.functional as F
from lightglue import ALIKED, LightGlue
from lightglue.utils import rbd
from skimage.exposure import match_histograms

PROJECT_ROOT = Path(os.environ.get("CAMERA_ARRAY_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
DATA_DIR = PROJECT_ROOT / "data"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Expected a data directory at {DATA_DIR}. Start Jupyter from the repository root "
        "or set CAMERA_ARRAY_PROJECT_ROOT to the cloned repository path."
    )
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARDUCAM_SHAPE = (3040, 4032)
VIMBA_SHAPE = (3036, 4024)
LUCID_SHAPE = (2048, 2448)
ARDUCAM_10BIT_WHITE_LEVEL = 1023.0
ARDUCAM_CONTAINER_WHITE_LEVEL = 65535.0
VIMBA_WHITE_LEVEL = 4095.0

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "STIXGeneral"],
    "font.size": 10,
    "figure.dpi": 120,
})
print(f"Project: {PROJECT_ROOT}\nCompute device: {DEVICE}")


## Shared utilities

These cells provide RAW loading, demosaicing, feature registration, overlap cropping, fusion, and visualization helpers used by the task sections below.


In [ ]:
def require_files(*paths):
    missing = [str(path) for path in paths if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError("Missing required data file(s):\n- " + "\n- ".join(missing))


def list_raw_files(directory):
    files = sorted(Path(directory).glob("*.raw"))
    if not files:
        raise FileNotFoundError(f"No .raw files found in {directory}")
    return files


def read_raw_exact(path, shape, dtype):
    path = Path(path)
    require_files(path)
    data = np.fromfile(path, dtype=dtype)
    expected = int(np.prod(shape))
    if data.size != expected:
        raise ValueError(f"{path.name}: expected {expected:,} samples, found {data.size:,}")
    return data.reshape(shape)


def malvar_demosaic_rggb(bayer):
    _, _, height, width = bayer.shape
    device = bayer.device
    y, x = torch.meshgrid(
        torch.arange(height, device=device),
        torch.arange(width, device=device),
        indexing="ij",
    )
    mask_r = ((y % 2 == 0) & (x % 2 == 0)).float()
    mask_b = ((y % 2 == 1) & (x % 2 == 1)).float()
    mask_gr = ((y % 2 == 0) & (x % 2 == 1)).float()
    mask_gb = ((y % 2 == 1) & (x % 2 == 0)).float()
    mask_g = mask_gr + mask_gb

    h_g = torch.tensor(
        [[0, 0, -1, 0, 0], [0, 0, 2, 0, 0], [-1, 2, 4, 2, -1],
         [0, 0, 2, 0, 0], [0, 0, -1, 0, 0]],
        dtype=torch.float32, device=device,
    ) / 8.0
    h_rb = torch.tensor(
        [[0, 0, -1.5, 0, 0], [0, 2, 0, 2, 0], [-1.5, 0, 6, 0, -1.5],
         [0, 2, 0, 2, 0], [0, 0, -1.5, 0, 0]],
        dtype=torch.float32, device=device,
    ) / 8.0
    h_rgr = torch.tensor(
        [[0, 0, 0.5, 0, 0], [0, -1, 0, -1, 0], [-1, 4, 5, 4, -1],
         [0, -1, 0, -1, 0], [0, 0, 0.5, 0, 0]],
        dtype=torch.float32, device=device,
    ) / 8.0
    weights = torch.stack([h_g, h_rb, h_rgr, h_rgr.T]).unsqueeze(1)

    with torch.inference_mode():
        convs = F.conv2d(F.pad(bayer, (2, 2, 2, 2), mode="reflect"), weights)
        c_g, c_rb, c_rgr, c_rgb = [value.unsqueeze(1) for value in convs.unbind(dim=1)]
        red = bayer * mask_r + c_rgr * mask_gr + c_rgb * mask_gb + c_rb * mask_b
        blue = bayer * mask_b + c_rgr * mask_gb + c_rgb * mask_gr + c_rb * mask_r
        green = bayer * mask_g + c_g * (mask_r + mask_b)
    return torch.clamp(torch.cat([red, green, blue], dim=1), 0.0, 1.0)


def load_arducam(path, white_level=None, r_gain=2.0, b_gain=2.0):
    raw = read_raw_exact(path, ARDUCAM_SHAPE, np.uint16).astype(np.float32)
    if white_level is None:
        white_level = (
            ARDUCAM_CONTAINER_WHITE_LEVEL
            if raw.max() > 4095
            else ARDUCAM_10BIT_WHITE_LEVEL
        )
    tensor = torch.from_numpy(raw).unsqueeze(0).unsqueeze(0).to(DEVICE) / white_level
    tensor[:, :, 0::2, 0::2] *= r_gain
    tensor[:, :, 1::2, 1::2] *= b_gain
    rgb = malvar_demosaic_rggb(torch.clamp(tensor, 0.0, 1.0))
    return rgb.squeeze(0).permute(1, 2, 0).cpu().numpy()


def load_vimba(path, white_level=VIMBA_WHITE_LEVEL):
    raw = read_raw_exact(path, VIMBA_SHAPE, np.uint16).astype(np.float32)
    return np.clip(raw / white_level, 0.0, 1.0)


def load_polarization(path):
    raw = read_raw_exact(path, LUCID_SHAPE, np.uint8).astype(np.float32) / 255.0
    i0, i45 = raw[0::2, 0::2], raw[0::2, 1::2]
    i90, i135 = raw[1::2, 0::2], raw[1::2, 1::2]
    s0 = 0.5 * (i0 + i45 + i90 + i135)
    s1, s2 = i0 - i90, i45 - i135
    dolp = np.clip(np.sqrt(s1**2 + s2**2) / (s0 + 1e-6), 0.0, 1.0)
    aolp = 0.5 * np.arctan2(s2, s1)
    return s0, dolp, aolp


In [ ]:
def _gray(image):
    return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if image.ndim == 3 else image


def _feature_tensor(image, gamma=None):
    if gamma is None:
        array = _gray(image).astype(np.float32)
        return torch.from_numpy(array).unsqueeze(0).unsqueeze(0).to(DEVICE)
    rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB) if image.ndim == 2 else image
    rgb = np.power(np.clip(rgb, 0.0, 1.0), gamma).astype(np.float32)
    return torch.from_numpy(rgb.transpose(2, 0, 1)).to(DEVICE)


def build_matcher(max_keypoints=4096):
    extractor = ALIKED(max_num_keypoints=max_keypoints).eval().to(DEVICE)
    matcher = LightGlue(features="aliked").eval().to(DEVICE)
    return extractor, matcher


def estimate_homography(
    source, target, max_keypoints=4096, threshold=5.0, min_matches=4, feature_gamma=None
):
    extractor, matcher = build_matcher(max_keypoints)
    with torch.inference_mode():
        source_features = extractor.extract(_feature_tensor(source, feature_gamma))
        target_features = extractor.extract(_feature_tensor(target, feature_gamma))
        result = matcher({"image0": source_features, "image1": target_features})
    source_features, target_features, result = [
        rbd(item) for item in (source_features, target_features, result)
    ]
    matches = result["matches"]
    source_points = source_features["keypoints"][matches[..., 0]].cpu().numpy().astype(np.float32)
    target_points = target_features["keypoints"][matches[..., 1]].cpu().numpy().astype(np.float32)
    if len(source_points) < min_matches:
        raise RuntimeError(
            f"Homography requires at least {min_matches} matches; found {len(source_points)}"
        )
    matrix, inliers = cv2.findHomography(
        source_points, target_points, cv2.USAC_MAGSAC, threshold
    )
    if matrix is None or inliers is None:
        raise RuntimeError("Homography estimation failed")
    return matrix, source_points, target_points, inliers


def ecc_align(source, target):
    source_gray = _gray(source).astype(np.float32)
    target_gray = _gray(target).astype(np.float32)
    matrix = np.eye(3, dtype=np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 50, 1e-4)
    _, matrix = cv2.findTransformECC(
        target_gray, source_gray, matrix, cv2.MOTION_HOMOGRAPHY, criteria
    )
    height, width = target.shape[:2]
    return cv2.warpPerspective(
        source,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
    )

def warp_to_target(source, target, matrix, interpolation=cv2.INTER_LINEAR):
    height, width = target.shape[:2]
    return cv2.warpPerspective(source, matrix, (width, height), flags=interpolation)


def crop_shared_area(warped_source, target, matrix, source_shape, erosion=25):
    mask = np.full(source_shape[:2], 255, dtype=np.uint8)
    target_height, target_width = target.shape[:2]
    warped_mask = cv2.warpPerspective(mask, matrix, (target_width, target_height))
    if erosion:
        kernel = np.ones((erosion, erosion), dtype=np.uint8)
        warped_mask = cv2.erode(warped_mask, kernel)
    x, y, width, height = cv2.boundingRect(warped_mask)
    if width == 0 or height == 0:
        raise RuntimeError("The registered images do not have a valid shared region")
    source_crop = warped_source[y:y + height, x:x + width]
    target_crop = target[y:y + height, x:x + width]
    return source_crop, target_crop, (x, y, width, height)


def mtf_glp_hpm_fusion(mono, color, sigma=1.5, match_intensity=False):
    mono_low = cv2.GaussianBlur(mono, (0, 0), sigmaX=sigma, sigmaY=sigma)
    mono_high = mono - mono_low
    base = color
    if match_intensity:
        intensity = np.mean(color, axis=2)
        base = color * (mono_low / (intensity + 1e-8))[..., None]
    gain = base / (mono_low[..., None] + 1e-8)
    return np.clip(base + mono_high[..., None] * gain, 0.0, 1.0)


def draw_image_grid(items, columns=3, figsize=(18, 10)):
    rows = int(np.ceil(len(items) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=figsize, squeeze=False)
    for axis, (title, image) in zip(axes.flat, items):
        axis.imshow(image, cmap="gray" if image.ndim == 2 else None, vmin=0, vmax=1)
        axis.set_title(title, fontweight="bold")
        axis.axis("off")
    for axis in axes.flat[len(items):]:
        axis.axis("off")
    plt.tight_layout()
    return fig


In [ ]:
def guided_filter(guide, source, radius=5, eps=1e-3):
    kernel = (radius, radius)
    mean_i = cv2.boxFilter(guide, cv2.CV_32F, kernel)
    mean_p = cv2.boxFilter(source, cv2.CV_32F, kernel)
    covariance = cv2.boxFilter(guide * source, cv2.CV_32F, kernel) - mean_i * mean_p
    variance = cv2.boxFilter(guide * guide, cv2.CV_32F, kernel) - mean_i * mean_i
    a = covariance / (variance + eps)
    b = mean_p - a * mean_i
    return cv2.boxFilter(a, cv2.CV_32F, kernel) * guide + cv2.boxFilter(b, cv2.CV_32F, kernel)


def fusion_comparison(color, mono):
    hsv = cv2.cvtColor(color, cv2.COLOR_RGB2HSV)
    hsv[..., 2] = mono

    lab = cv2.cvtColor(color, cv2.COLOR_RGB2LAB)
    lab[..., 0] = mono * 100.0

    intensity = np.mean(color, axis=2)
    brovey = color / (intensity[..., None] + 1e-6) * mono[..., None]

    hsv_matched = cv2.cvtColor(color, cv2.COLOR_RGB2HSV)
    hsv_matched[..., 2] = match_histograms(mono, hsv_matched[..., 2])

    flat = color.reshape(-1, 3)
    mean = flat.mean(axis=0)
    covariance = np.cov((flat - mean).T)
    _, eigenvectors = np.linalg.eigh(covariance)
    eigenvectors = eigenvectors[:, ::-1]
    components = (flat - mean) @ eigenvectors
    components[:, 0] = match_histograms(mono.ravel(), components[:, 0])
    pca = (components @ eigenvectors.T + mean).reshape(color.shape)

    hsv_dwt = cv2.cvtColor(color, cv2.COLOR_RGB2HSV)
    ll_v, details_v = pywt.dwt2(hsv_dwt[..., 2], "haar")
    _, details_m = pywt.dwt2(mono, "haar")
    hsv_dwt[..., 2] = pywt.idwt2((ll_v, details_m), "haar")[: mono.shape[0], : mono.shape[1]]

    guided = np.stack([guided_filter(mono, color[..., channel]) for channel in range(3)], axis=2)
    return {
        "Aligned color": color,
        "Monochrome": mono,
        "HSV substitution": np.clip(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB), 0, 1),
        "Lab substitution": np.clip(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB), 0, 1),
        "Brovey": np.clip(brovey, 0, 1),
        "Histogram-matched HSV": np.clip(cv2.cvtColor(hsv_matched, cv2.COLOR_HSV2RGB), 0, 1),
        "PCA": np.clip(pca, 0, 1),
        "DWT": np.clip(cv2.cvtColor(hsv_dwt, cv2.COLOR_HSV2RGB), 0, 1),
        "Guided filter": np.clip(guided, 0, 1),
        "MTF-GLP-HPM": mtf_glp_hpm_fusion(mono, color),
    }


# Task 1A — 21 mm color to 50 mm monochrome fusion comparison

Registers the 21 mm Arducam frame to the 50 mm Vimba frame, then compares eight intensity/detail-fusion methods. Change only the two filenames below when evaluating another pair.


In [ ]:
TASK1_DIR = DATA_DIR / "paired_frames"
TASK1_COLOR = TASK1_DIR / "row4_cam3_arducam_21mm_f0008.raw"
TASK1_MONO = TASK1_DIR / "row3_cam8_vimba_50mm_f0008.raw"
require_files(TASK1_COLOR, TASK1_MONO)

task1_color = load_arducam(TASK1_COLOR)
task1_mono = load_vimba(TASK1_MONO)
task1_h, task1_color_points, task1_mono_points, task1_inliers = estimate_homography(
    task1_color, task1_mono
)
task1_aligned = warp_to_target(task1_color, task1_mono, task1_h)
print(f"Matches: {len(task1_color_points)} | Inliers: {int(task1_inliers.sum())}")


In [ ]:
task1_results = fusion_comparison(task1_aligned.astype(np.float32), task1_mono.astype(np.float32))
draw_image_grid(list(task1_results.items()), columns=5, figsize=(24, 12));


# Task 1B — Overlap-cropped 21 mm color to 25 mm monochrome fusion

This is the separate workflow that defines the original `overlap_color`, `overlap_mono`, `ov_w`, `ov_h`, and zoom-region variables. It crops both registered images to their shared field of view before fusion and performs local homography refinement inside a configurable patch.


In [ ]:
TASK1B_DIR = DATA_DIR / "paired_frames"
TASK1B_COLOR = TASK1B_DIR / "row3_cam4_arducam_21mm_f0008.raw"
TASK1B_MONO = TASK1B_DIR / "row3_cam7_vimba_25mm_f0008.raw"
TASK1B_PATCH_SIZE = 400
TASK1B_PATCH_XY = (2500, 2000)
require_files(TASK1B_COLOR, TASK1B_MONO)

task1b_color = load_arducam(TASK1B_COLOR)
task1b_mono = load_vimba(TASK1B_MONO)
task1b_h, task1b_color_points, task1b_mono_points, task1b_inliers = estimate_homography(
    task1b_color, task1b_mono
)
task1b_aligned = warp_to_target(task1b_color, task1b_mono, task1b_h)
task1b_overlap_color, task1b_overlap_mono, task1b_crop_box = crop_shared_area(
    task1b_aligned, task1b_mono, task1b_h, task1b_color.shape, erosion=0
)
task1b_ov_h, task1b_ov_w = task1b_overlap_mono.shape
task1b_results = fusion_comparison(task1b_overlap_color, task1b_overlap_mono)

print(
    f"Matches: {len(task1b_color_points)} | Inliers: {int(task1b_inliers.sum())} | "
    f"Shared field: {task1b_ov_w} × {task1b_ov_h}"
)
draw_image_grid(list(task1b_results.items()), columns=5, figsize=(24, 12));


In [ ]:
if min(task1b_ov_h, task1b_ov_w) < TASK1B_PATCH_SIZE:
    raise ValueError(
        f"The shared field ({task1b_ov_w} × {task1b_ov_h}) is smaller than "
        f"the requested {TASK1B_PATCH_SIZE}-pixel patch"
    )

task1b_patch_x = int(np.clip(
    TASK1B_PATCH_XY[0], 0, max(0, task1b_ov_w - TASK1B_PATCH_SIZE)
))
task1b_patch_y = int(np.clip(
    TASK1B_PATCH_XY[1], 0, max(0, task1b_ov_h - TASK1B_PATCH_SIZE)
))
task1b_patch_slice = np.s_[
    task1b_patch_y:task1b_patch_y + TASK1B_PATCH_SIZE,
    task1b_patch_x:task1b_patch_x + TASK1B_PATCH_SIZE,
]
task1b_color_patch = task1b_overlap_color[task1b_patch_slice].copy()
task1b_mono_patch = task1b_overlap_mono[task1b_patch_slice].copy()
task1b_global_patch = task1b_results["MTF-GLP-HPM"][task1b_patch_slice].copy()

try:
    task1b_local_h, task1b_local_color_points, task1b_local_mono_points, _ = estimate_homography(
        task1b_color_patch, task1b_mono_patch, threshold=2.0, min_matches=10
    )
    task1b_refined_color = cv2.warpPerspective(
        task1b_color_patch,
        task1b_local_h,
        (TASK1B_PATCH_SIZE, TASK1B_PATCH_SIZE),
    )
    print(f"Local matches: {len(task1b_local_color_points)}")
except (RuntimeError, cv2.error) as error:
    print(f"Using ECC local refinement: {error}")
    try:
        task1b_refined_color = ecc_align(task1b_color_patch, task1b_mono_patch)
    except cv2.error as ecc_error:
        print(f"ECC refinement skipped: {ecc_error}")
        task1b_refined_color = task1b_color_patch

task1b_local_fused = mtf_glp_hpm_fusion(task1b_mono_patch, task1b_refined_color)

fig, axes = plt.subplots(2, 2, figsize=(14, 13))
panels = [
    ("Globally aligned color patch", task1b_color_patch),
    ("Locally refined color patch", task1b_refined_color),
    ("Global MTF-GLP-HPM patch", task1b_global_patch),
    ("Locally refined MTF-GLP-HPM patch", task1b_local_fused),
]
for axis, (title, image) in zip(axes.flat, panels):
    axis.imshow(image, vmin=0, vmax=1)
    axis.set_title(title, fontweight="bold")
    axis.axis("off")
plt.tight_layout()


# Task 2 — Polarization registration and structural overlays

Registers Lucid polarization measurements to the 4 mm Arducam RGB frame. The first result uses one global homography; the second uses a 3 × 3 piecewise model with a global fallback for low-texture cells.


In [ ]:
TASK2_DIR = DATA_DIR / "polarization"
TASK2_COLOR = TASK2_DIR / "row3_cam3_arducam_4mm_f0008.raw"
TASK2_POLARIZATION = TASK2_DIR / "row1_cam7_lucid_pol_f0008.raw"
require_files(TASK2_COLOR, TASK2_POLARIZATION)

task2_color = load_arducam(TASK2_COLOR)
task2_s0, task2_dolp, task2_aolp = load_polarization(TASK2_POLARIZATION)
task2_h, task2_pol_points, task2_rgb_points, task2_inliers = estimate_homography(
    task2_s0, task2_color, max_keypoints=12000
)
print(f"Matches: {len(task2_pol_points)} | Inliers: {int(task2_inliers.sum())}")


In [ ]:
def polarization_overlay(color, warped_s0, warped_dolp, active_mask):
    s0_u8 = (np.clip(warped_s0, 0, 1) * 255).astype(np.uint8)
    dolp_u8 = (np.clip(warped_dolp, 0, 1) * 255).astype(np.uint8)
    s0_smooth = cv2.bilateralFilter(s0_u8, 9, 75, 75)
    dolp_smooth = cv2.bilateralFilter(dolp_u8, 9, 75, 75)
    edges = cv2.bitwise_or(cv2.Canny(s0_smooth, 30, 90), cv2.Canny(dolp_smooth, 20, 60))
    edges = cv2.bitwise_and(edges, active_mask)
    edges = cv2.dilate(edges, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)), iterations=2)
    overlay = color.copy()
    overlay[edges == 255] = [0.0, 1.0, 1.0]
    return overlay


task2_warped_s0 = warp_to_target(task2_s0, task2_color, task2_h)
task2_warped_dolp = warp_to_target(task2_dolp, task2_color, task2_h)
task2_mask = warp_to_target(np.full_like(task2_s0, 255, dtype=np.uint8), task2_color, task2_h)
task2_mask = cv2.erode(task2_mask, cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15)))
task2_global_overlay = polarization_overlay(
    task2_color, task2_warped_s0, task2_warped_dolp, task2_mask
)


In [ ]:
def piecewise_polarization_warp(
    s0, dolp, target_shape, source_points, target_points, global_h, grid=3
):
    height, width = target_shape[:2]
    warped_s0 = np.zeros((height, width), dtype=np.float32)
    warped_dolp = np.zeros((height, width), dtype=np.float32)
    active_mask = np.zeros((height, width), dtype=np.uint8)
    x_edges = np.linspace(0, width, grid + 1, dtype=int)
    y_edges = np.linspace(0, height, grid + 1, dtype=int)

    for row in range(grid):
        for column in range(grid):
            x0, x1 = x_edges[column], x_edges[column + 1]
            y0, y1 = y_edges[row], y_edges[row + 1]
            local = (
                (target_points[:, 0] >= x0 - 250) & (target_points[:, 0] < x1 + 250)
                & (target_points[:, 1] >= y0 - 250) & (target_points[:, 1] < y1 + 250)
            )
            matrix = global_h
            if local.sum() >= 15:
                candidate, _ = cv2.findHomography(
                    source_points[local], target_points[local], cv2.USAC_MAGSAC, 4.0
                )
                if candidate is not None:
                    matrix = candidate
            local_s0 = cv2.warpPerspective(s0, matrix, (width, height))
            local_dolp = cv2.warpPerspective(dolp, matrix, (width, height))
            local_mask = cv2.warpPerspective(
                np.full_like(s0, 255, dtype=np.uint8), matrix, (width, height)
            )
            warped_s0[y0:y1, x0:x1] = local_s0[y0:y1, x0:x1]
            warped_dolp[y0:y1, x0:x1] = local_dolp[y0:y1, x0:x1]
            active_mask[y0:y1, x0:x1] = local_mask[y0:y1, x0:x1]

    active_mask = cv2.erode(active_mask, cv2.getStructuringElement(cv2.MORPH_RECT, (21, 21)))
    return warped_s0, warped_dolp, active_mask


task2_mesh_s0, task2_mesh_dolp, task2_mesh_mask = piecewise_polarization_warp(
    task2_s0,
    task2_dolp,
    task2_color.shape,
    task2_pol_points,
    task2_rgb_points,
    task2_h,
)
task2_mesh_overlay = polarization_overlay(
    task2_color, task2_mesh_s0, task2_mesh_dolp, task2_mesh_mask
)

draw_image_grid([
    ("Arducam RGB", task2_color),
    ("Global polarization overlay", task2_global_overlay),
    ("3 × 3 piecewise overlay", task2_mesh_overlay),
], columns=3, figsize=(21, 7));


# Task 3 — Exposure fusion, focus stacking, and cross-modal fusion

Combines the Vimba exposure sweep with Mertens fusion, combines the Arducam focal sweep using a Laplacian focus measure, registers the two results, and performs MTF-GLP-HPM fusion.


In [ ]:
def mertens_mono_fusion(images):
    if not images:
        raise ValueError("At least one exposure is required")
    rgb_images = [cv2.cvtColor(image.astype(np.float32), cv2.COLOR_GRAY2RGB) for image in images]
    merger = cv2.createMergeMertens(1.0, 1.0, 1.0)
    fused = merger.process(rgb_images)
    mono = cv2.cvtColor(fused.astype(np.float32), cv2.COLOR_RGB2GRAY)
    return cv2.normalize(mono, None, 0, 1, cv2.NORM_MINMAX)


def laplacian_focus_stack(images, blur_size=75, softmax_scale=50.0):
    if not images:
        raise ValueError("At least one focus frame is required")
    if blur_size % 2 == 0:
        raise ValueError("blur_size must be odd")
    measures = []
    for image in images:
        gray = cv2.cvtColor((image * 255).astype(np.float32), cv2.COLOR_RGB2GRAY)
        laplacian = cv2.Laplacian(gray, cv2.CV_32F, ksize=5)
        measures.append(cv2.GaussianBlur(np.abs(laplacian), (blur_size, blur_size), 0))
    measures = np.stack(measures)
    shifted = measures - measures.max(axis=0, keepdims=True)
    weights = np.exp(shifted * softmax_scale)
    weights /= weights.sum(axis=0, keepdims=True) + 1e-12
    stack = sum(image * weights[index, ..., None] for index, image in enumerate(images))
    return np.clip(stack, 0.0, 1.0)


In [ ]:
TASK3_ARDUCAM_DIR = DATA_DIR / "sweeps" / "arducam"
TASK3_VIMBA_DIR = DATA_DIR / "sweeps" / "vimba"
TASK3_RGB_GAINS = np.array([0.85, 1.0, 0.90], dtype=np.float32)
task3_arducam_files = list_raw_files(TASK3_ARDUCAM_DIR)
task3_vimba_files = list_raw_files(TASK3_VIMBA_DIR)

task3_color_frames = [load_arducam(path) for path in task3_arducam_files]
task3_mono_frames = [load_vimba(path) for path in task3_vimba_files]
print(f"Loaded {len(task3_color_frames)} focus frames and {len(task3_mono_frames)} exposure frames")


In [ ]:
task3_hdr_mono = mertens_mono_fusion(task3_mono_frames)
task3_focus_color = laplacian_focus_stack(task3_color_frames)
task3_h, task3_color_points, task3_mono_points, task3_inliers = estimate_homography(
    task3_focus_color, task3_hdr_mono, threshold=3.0, feature_gamma=1.0 / 2.2
)
task3_aligned_color = np.clip(
    warp_to_target(task3_focus_color, task3_hdr_mono, task3_h, cv2.INTER_CUBIC),
    0.0,
    1.0,
)
task3_fused = mtf_glp_hpm_fusion(
    task3_hdr_mono, task3_aligned_color, match_intensity=True
)
task3_fused = np.clip(task3_fused * TASK3_RGB_GAINS, 0.0, 1.0)

draw_image_grid([
    ("Vimba Mertens exposure fusion", task3_hdr_mono),
    ("Arducam Laplacian focus stack", task3_focus_color),
    ("Registered color stack", task3_aligned_color),
    ("MTF-GLP-HPM result", task3_fused),
], columns=2, figsize=(16, 13));


# Task 4 — Synchronized-pair local refinement

Selects a synchronized RGB/mono pair by matching filename, performs global alignment and overlap cropping, refines one local patch, and compares global and locally refined fusion results.


In [ ]:
TASK4_COLOR_DIR = DATA_DIR / "synchronized" / "color"
TASK4_MONO_DIR = DATA_DIR / "synchronized" / "mono"
TASK4_PAIR_INDEX = 0
TASK4_PATCH_SIZE = 400
TASK4_PATCH_OFFSET = (-850, -150)

task4_color_lookup = {path.name: path for path in list_raw_files(TASK4_COLOR_DIR)}
task4_mono_lookup = {path.name: path for path in list_raw_files(TASK4_MONO_DIR)}
task4_names = sorted(task4_color_lookup.keys() & task4_mono_lookup.keys())
if not task4_names:
    raise FileNotFoundError("No synchronized color/mono filenames match")
if not 0 <= TASK4_PAIR_INDEX < len(task4_names):
    raise IndexError(f"TASK4_PAIR_INDEX must be between 0 and {len(task4_names) - 1}")

task4_name = task4_names[TASK4_PAIR_INDEX]
task4_color = load_arducam(task4_color_lookup[task4_name])
task4_mono = load_vimba(task4_mono_lookup[task4_name])
print(f"Selected synchronized pair: {task4_name}")


In [ ]:
task4_h, task4_color_points, task4_mono_points, task4_inliers = estimate_homography(
    task4_color, task4_mono
)
task4_aligned_full = warp_to_target(task4_color, task4_mono, task4_h)
task4_color_crop, task4_mono_crop, task4_crop_box = crop_shared_area(
    task4_aligned_full, task4_mono, task4_h, task4_color.shape
)
task4_global_fused = mtf_glp_hpm_fusion(task4_mono_crop, task4_color_crop)

crop_height, crop_width = task4_mono_crop.shape
if min(crop_height, crop_width) < TASK4_PATCH_SIZE:
    raise ValueError(
        f"The shared field ({crop_width} × {crop_height}) is smaller than "
        f"the requested {TASK4_PATCH_SIZE}-pixel patch"
    )
offset_x, offset_y = TASK4_PATCH_OFFSET
patch_x = int(np.clip(
    crop_width // 2 - TASK4_PATCH_SIZE // 2 + offset_x,
    0,
    crop_width - TASK4_PATCH_SIZE,
))
patch_y = int(np.clip(
    crop_height // 2 - TASK4_PATCH_SIZE // 2 + offset_y,
    0,
    crop_height - TASK4_PATCH_SIZE,
))
patch_slice = np.s_[patch_y:patch_y + TASK4_PATCH_SIZE, patch_x:patch_x + TASK4_PATCH_SIZE]
task4_mono_patch = task4_mono_crop[patch_slice].copy()
task4_color_patch = task4_color_crop[patch_slice].copy()
task4_global_patch = task4_global_fused[patch_slice].copy()

blurred_mono = cv2.GaussianBlur(task4_mono_patch, (19, 19), 3.0)
try:
    task4_local_h, task4_local_color_points, task4_local_mono_points, _ = estimate_homography(
        task4_color_patch, blurred_mono, threshold=2.0
    )
    task4_aligned_patch = cv2.warpPerspective(
        task4_color_patch, task4_local_h, (TASK4_PATCH_SIZE, TASK4_PATCH_SIZE)
    )
except (RuntimeError, cv2.error) as error:
    print(f"Local refinement skipped: {error}")
    task4_aligned_patch = task4_color_patch

task4_local_fused = mtf_glp_hpm_fusion(task4_mono_patch, task4_aligned_patch)
print(
    f"Global matches: {len(task4_color_points)} | "
    f"Crop: {task4_crop_box[2]} × {task4_crop_box[3]}"
)


In [ ]:
draw_image_grid([
    ("Local monochrome patch", task4_mono_patch),
    ("Globally aligned color patch", task4_color_patch),
    ("Global MTF-GLP-HPM fusion", task4_global_patch),
    ("Locally refined MTF-GLP-HPM fusion", task4_local_fused),
], columns=4, figsize=(20, 5));


## Notes for release

- Keep RAW data filenames unchanged or update the short configuration cell at the start of each task.
- `requirements.txt` pins the environment used for real-data validation. Add the data license and permanent download location to `data/README.md` before archival.
